# 02 · Analysis table: London LSOAs
One row per 2011 LSOA with the 2016 prescribing outcomes, the nine greenery measures and the four controls (Census 2011). This is the table every model reads.

**Inputs:** `data/prescriptions/lsoa_prescriptions_2016_*.csv`, `data/greenery_metrics/*_lsoa.csv`, `data/greenery_metrics/official_indices/lsoas_official_indices.csv`, `data/raw_data/controls/*`, `data/raw_data/density_data/extracted_per_LSOA.csv`, LSOA boundaries.
**Outputs:** `data/data_analysis_lsoas7s.csv` (controls as described below; greenery and outcomes in their own units, not standardised; the models standardise per sample) and `data/data_analysis_lsoas7s_geo.geojson`.
**Runtime:** under a minute. Run from this folder.

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # run from this folder
DATA = REPO / 'data'
RAW = DATA / 'raw_data'
GM = DATA / 'greenery_metrics'
LOOKUP = pd.read_csv(RAW / 'lookups/Lower_Layer_Super_Output_Area_(2011)_to_Ward_(2018)_Lookup_in_England_and_Wales_v3.csv')
OUTCOMES = ['diabetes', 'hypertension', 'asthma', 'opioids', 'total']
LEVEL, KEY = 'lsoa', 'lsoa_code'

## 1. Prescribing: yearly quantity per registered patient, 2016

In [2]:
data = None
for o in OUTCOMES:
    df = pd.read_csv(DATA / f'prescriptions/lsoa_prescriptions_2016_{o}.csv')
    df = df[[ 'LSOA11CD', 'Total_quantity_per_capita']].rename(columns={'LSOA11CD': KEY, 'Total_quantity_per_capita': f'{o} prescriptions'})
    data = df if data is None else data.merge(df, on=KEY)
print(len(data), 'LSOAs')

4835 LSOAs


## 2. Greenery measures
NDVI and our five measures are shares (0–1) of the area, road strip or street-view pixels; the official measures are shares of residents with access.

In [3]:
# file stem -> {source column: name in the analysis table}
SOURCES = {
    'SAT_greenery_on_off_road': {'on_road_green_share': 'SAT_on_road_share_u', 'on_road_green_share_weighted': 'SAT_on_road_share_w'},
    'SAT_greenery_off_road_accessibility': {'accessible_off_road_green': 'SAT_off_road'},
    'GSV_greenery_per': {'GSV_on_road': 'GSV_on_road_u', 'GSV_on_road_weighted': 'GSV_on_road_w'},
}
greenery = pd.read_csv(GM / f'SAT_greenery_total_{LEVEL}.csv')[[KEY, 'percent_green']].rename(columns={'percent_green': 'SAT_NDVI'})
for stem, cols in SOURCES.items():
    df = pd.read_csv(GM / f'{stem}_{LEVEL}.csv')
    greenery = greenery.merge(df[[KEY, *cols]].rename(columns=cols), on=KEY, how='left')
assert greenery[KEY].is_unique
data = data.merge(greenery, on=KEY)
print(len(data), 'areas with prescribing and greenery')
official = pd.read_csv(GM / 'official_indices/lsoas_official_indices.csv').rename(columns={'LSOA11CD': KEY})
# left join: about 190 LSOAs have no official values (no population grid cell); models drop them per measure
data = data.merge(official[[KEY, 'WHO', 'ESA_WHO_exposure', 'Natural_England1']], on=KEY, how='left')
print(data['WHO'].isna().sum(), 'LSOAs without official measures')

4835 areas with prescribing and greenery
193 LSOAs without official measures


## 3. Controls (Census 2011)
- `median age`: the LSOA median.
- `white percent`: share of residents who are White.
- `IMD deprivation score`: share of households deprived in 2 of the 4 Census dimensions (a Census measure, not the Index of Multiple Deprivation; the name is kept for the model code).
- `building_density`: mean relative building height (`relh2`) from the GLA building footprints, log(x + 1).

In [4]:
age = pd.read_csv(RAW / 'controls/age.csv')
eth = pd.read_csv(RAW / 'controls/ethnic_group.csv')
dep = pd.read_csv(RAW / 'controls/IMD.csv')
controls = pd.DataFrame({
    KEY: age['geography code'],
    'median age': age['Age: Median Age; measures: Value'],
}).merge(pd.DataFrame({
    KEY: eth['geography code'],
    'white percent': eth['Ethnic Group: White; measures: Value'] / eth['Ethnic Group: All usual residents; measures: Value'],
}), on=KEY).merge(pd.DataFrame({
    KEY: dep['geography code'],
    'IMD deprivation score': dep['Household Deprivation: Household is deprived in 2 dimensions; measures: Value']
    / dep['Household Deprivation: All categories: Classification of household deprivation; measures: Value'],
}), on=KEY)
height = pd.read_csv(RAW / 'density_data/extracted_per_LSOA.csv')[['lsoa_code', 'relh2']]
data = data.merge(controls, on=KEY)
data = data.merge(height.rename(columns={'relh2': 'building_density'}), on=KEY, how='left')
missing = data.loc[data['building_density'].isna(), KEY].tolist()
print(f'{len(missing)} LSOAs without building height, dropped: {missing}')
data = data.dropna(subset=['building_density'])
data['building_density'] = np.log(data['building_density'] + 1)

0 LSOAs without building height, dropped: []


## 4. Centroids (British National Grid metres, for MGWR) → SAVE

In [5]:
geo = gpd.read_file(RAW / 'geo_LSOAs/LSOA_(Dec_2011)_Boundaries_Generalised_Clipped_(BGC)_EW_V3.shp')[['LSOA11CD', 'geometry']].rename(columns={'LSOA11CD': KEY})
cent = geo.geometry.centroid
data = data.merge(geo[[KEY]].assign(centroid_lon=cent.x, centroid_lat=cent.y), on=KEY)
assert data[KEY].is_unique and len(data) == 4835, len(data)
data.to_csv(DATA / 'data_analysis_lsoas7s.csv', index=False)
gpd.GeoDataFrame(data.merge(geo, on=KEY), geometry='geometry', crs=geo.crs).to_file(DATA / 'data_analysis_lsoas7s_geo.geojson', driver='GeoJSON')
print(len(data), 'LSOAs written;', list(data.columns))

4835 LSOAs written; ['lsoa_code', 'diabetes prescriptions', 'hypertension prescriptions', 'asthma prescriptions', 'opioids prescriptions', 'total prescriptions', 'SAT_NDVI', 'SAT_on_road_share_u', 'SAT_on_road_share_w', 'SAT_off_road', 'GSV_on_road_u', 'GSV_on_road_w', 'WHO', 'ESA_WHO_exposure', 'Natural_England1', 'median age', 'white percent', 'IMD deprivation score', 'building_density', 'centroid_lon', 'centroid_lat']


## Validation

In [6]:
data.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
diabetes prescriptions,4835.0,50.986,17.042,6.841,39.140,49.531,60.343,122.858
hypertension prescriptions,4835.0,85.582,19.634,16.723,70.174,85.131,99.755,206.697
asthma prescriptions,4835.0,14.147,2.674,3.630,12.183,13.978,15.943,40.736
opioids prescriptions,4835.0,14.322,4.788,1.466,10.975,13.523,17.046,38.155
total prescriptions,4835.0,1242.981,265.920,247.780,1073.197,1256.391,1406.625,6338.427
SAT_NDVI,4835.0,0.375,0.162,0.016,0.260,0.353,0.468,0.929
SAT_on_road_share_u,4834.0,0.218,0.097,0.000,0.150,0.203,0.269,0.870
SAT_on_road_share_w,4834.0,0.201,0.091,0.003,0.135,0.185,0.250,0.812
SAT_off_road,4835.0,0.052,0.105,0.000,0.000,0.004,0.055,0.825
GSV_on_road_u,4817.0,0.074,0.029,0.005,0.053,0.071,0.093,0.188
